# 6 - TF-IDF + Logistic Regression


## 1. Install dan setup

In [1]:
!pip install -q pandas numpy scikit-learn matplotlib openpyxl joblib tqdm

# =========================================================
# Helper: input fleksibel untuk Google Colab
# =========================================================
from pathlib import Path
import os, re, shutil, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
EDA_DIR = OUTPUT_DIR / "eda"
SENTIMENT_DIR = OUTPUT_DIR / "sentiment"
MODEL_DIR = OUTPUT_DIR / "models"
FINAL_DIR = OUTPUT_DIR / "final"

for d in [DATA_DIR, OUTPUT_DIR, EDA_DIR, SENTIMENT_DIR, MODEL_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def upload_file_colab(prompt="Upload file CSV"):
    """Upload file di Google Colab dan return Path file pertama."""
    from google.colab import files
    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Tidak ada file yang di-upload.")
    filename = list(uploaded.keys())[0]
    return Path(filename)


def find_or_upload(candidates, upload_prompt):
    """
    Cari file dari daftar kandidat. Kalau tidak ada, minta upload.
    Cocok untuk Colab: bisa lanjut dari file hasil notebook sebelumnya.
    """
    for p in candidates:
        p = Path(p)
        if p.exists():
            print(f"File ditemukan: {p}")
            return p
    return upload_file_colab(upload_prompt)


def normalize_sentiment_label(label):
    """Samakan label menjadi Positive / Neutral / Negative."""
    if pd.isna(label):
        return np.nan
    text = str(label).strip().lower()

    positive_values = {"positive", "positif", "pos", "1", "label_0"}
    neutral_values = {"neutral", "netral", "neu", "0", "label_1"}
    negative_values = {"negative", "negatif", "neg", "-1", "label_2"}

    if text in positive_values:
        return "Positive"
    if text in neutral_values:
        return "Neutral"
    if text in negative_values:
        return "Negative"
    return np.nan


def standardize_dataframe(df):
    """
    Standarisasi kolom dari hasil preprocessing 2a.
    File sumber biasanya punya: Link, Judul, Isi Berita, Status, Tag, Sentimen, Penerbit,
    final_text_clean, final_text_stemmed, jumlah_token_before_sw, jumlah_token_after_sw.
    """
    df = df.copy()

    rename_map = {
        "Link": "source_url",
        "Judul": "title",
        "Isi Berita": "content",
        "Status": "scrape_status",
        "Tag": "tag",
        "Sentimen": "sentiment_original",
        "Sentiment": "sentiment_original",
        "Penerbit": "publisher",
        "Tanggal": "published_date"
    }

    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]

    # Label asli/manual untuk evaluasi
    if "label_true" not in df.columns:
        if "sentiment_original" in df.columns:
            df["label_true"] = df["sentiment_original"].apply(normalize_sentiment_label)
        else:
            df["label_true"] = np.nan

    # Kolom teks fallback
    if "final_text_clean" not in df.columns:
        if "content" in df.columns:
            df["final_text_clean"] = df["content"].fillna("").astype(str).str.lower()
        else:
            raise KeyError("Tidak ditemukan kolom final_text_clean atau content/Isi Berita.")

    if "final_text_stemmed" not in df.columns:
        df["final_text_stemmed"] = df["final_text_clean"]

    # Token count fallback
    if "jumlah_token_before_sw" not in df.columns:
        df["jumlah_token_before_sw"] = df["final_text_clean"].fillna("").astype(str).str.split().apply(len)
    if "jumlah_token_after_sw" not in df.columns:
        df["jumlah_token_after_sw"] = df["final_text_stemmed"].fillna("").astype(str).str.split().apply(len)

    # Filter success kalau kolom status ada
    if "scrape_status" in df.columns:
        success_mask = df["scrape_status"].astype(str).str.lower().eq("success")
        if success_mask.sum() > 0:
            df = df[success_mask].copy()

    # Buang teks kosong
    df["final_text_clean"] = df["final_text_clean"].fillna("").astype(str)
    df["final_text_stemmed"] = df["final_text_stemmed"].fillna("").astype(str)
    df = df[df["final_text_clean"].str.strip().ne("")].copy()
    df = df.reset_index(drop=True)

    return df


def load_after_2a_csv(path):
    """Load CSV hasil preprocessing 2a dengan fallback encoding."""
    path = Path(path)
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")
    return standardize_dataframe(df)


def safe_display(obj, n=5):
    try:
        display(obj)
    except Exception:
        print(obj.head(n) if hasattr(obj, "head") else obj)


## 2. Input data
Gunakan hasil TextBlob jika ada, atau upload file hasil TextBlob/preprocessing.

In [2]:

INPUT_PATH = find_or_upload(
    candidates=[
        SENTIMENT_DIR / "hasil_textblob_sentiment.csv",
        "/content/hasil_textblob_sentiment.csv",
        DATA_DIR / "hasil_preprocessing_pertamina_standardized.csv",
        "/content/hasil_preprocessing_pertamina.csv",
    ],
    upload_prompt="Upload hasil_textblob_sentiment.csv atau hasil_preprocessing_pertamina.csv"
)

df = load_after_2a_csv(INPUT_PATH)
# Kalau file input sudah punya hasil TextBlob, standardize_dataframe tetap mempertahankan kolom tersebut.
print("Input:", INPUT_PATH)
print("Shape:", df.shape)
print("Distribusi label_true:")
safe_display(df["label_true"].value_counts(dropna=False).reset_index())
safe_display(df.head(3))


Upload hasil_textblob_sentiment.csv atau hasil_preprocessing_pertamina.csv


Saving hasil_preprocessing_pertamina.csv to hasil_preprocessing_pertamina.csv
Input: hasil_preprocessing_pertamina.csv
Shape: (1829, 19)
Distribusi label_true:


,label_true,count
0,Positive,765
1,Neutral,657
2,Negative,407


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit,final_text_clean,final_text_stemmed,jumlah_token_before_sw,jumlah_token_after_sw,source_url,title,content,scrape_status,tag,sentiment_original,publisher,label_true
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Neutral,Kompas,diskusi mengenai csr industri pelumas bertajuk...,diskusi kena csr industri lumas tajuk cari for...,16,13,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Neutral,Kompas,Neutral
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positive,Kompas,jakarta kompas com anak perusahaan pertamina y...,anak usaha pertamina pertamina lubricants gand...,130,95,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positive,Kompas,Positive
2,https://money.kompas.com/read/2017/02/03/12195...,"Dua Pucuk Pimpinan Pertamina Dicopot, Yenni An...","JAKARTA, KOMPAS.com — Pasca-pencopotan dua puc...",success,Migas,Positive,Kompas,jakarta kompas com pasca pencopotan dua pucuk ...,pasca copot pucuk pimpin pertamina menteri bad...,145,120,https://money.kompas.com/read/2017/02/03/12195...,"Dua Pucuk Pimpinan Pertamina Dicopot, Yenni An...","JAKARTA, KOMPAS.com — Pasca-pencopotan dua puc...",success,Migas,Positive,Kompas,Positive


## 3. TF-IDF + Logistic Regression

In [3]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import joblib

TEXT_COL_LOGREG = "final_text_stemmed"
LABEL_COL = "label_true"

labeled_df = df.dropna(subset=[LABEL_COL]).copy()
print("Jumlah data berlabel:", len(labeled_df))

if labeled_df[LABEL_COL].nunique() < 2:
    raise ValueError("Label hanya memiliki kurang dari 2 kelas. Logistic Regression tidak bisa dilatih.")

label_counts = labeled_df[LABEL_COL].value_counts()
can_stratify = label_counts.min() >= 2
print("Distribusi label training/evaluasi:")
safe_display(label_counts.reset_index())

idx_train, idx_test = train_test_split(
    labeled_df.index,
    test_size=0.2,
    random_state=42,
    stratify=labeled_df[LABEL_COL] if can_stratify else None
)

X_train = df.loc[idx_train, TEXT_COL_LOGREG]
y_train = df.loc[idx_train, LABEL_COL]
X_test = df.loc[idx_test, TEXT_COL_LOGREG]
y_test = df.loc[idx_test, LABEL_COL]

logreg_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42,
        solver="lbfgs"
    ))
])

logreg_model.fit(X_train, y_train)

# Prediksi seluruh data untuk dibandingkan nanti
df["logreg_sentiment"] = logreg_model.predict(df[TEXT_COL_LOGREG])
df["split"] = "unlabeled_or_not_tested"
df.loc[idx_train, "split"] = "train"
df.loc[idx_test, "split"] = "test"

# Probabilitas per kelas
if hasattr(logreg_model.named_steps["clf"], "classes_"):
    classes = list(logreg_model.named_steps["clf"].classes_)
    proba = logreg_model.predict_proba(df[TEXT_COL_LOGREG])
    for i, cls in enumerate(classes):
        df[f"logreg_prob_{cls}"] = proba[:, i]
    df["logreg_confidence"] = proba.max(axis=1)

# Evaluasi test set
y_pred_test = logreg_model.predict(X_test)
print("Classification Report - Logistic Regression Test Set")
print(classification_report(y_test, y_pred_test, labels=["Positive", "Neutral", "Negative"], zero_division=0))

cm_logreg = pd.DataFrame(
    confusion_matrix(y_test, y_pred_test, labels=["Positive", "Neutral", "Negative"]),
    index=["Actual_Positive", "Actual_Neutral", "Actual_Negative"],
    columns=["Pred_Positive", "Pred_Neutral", "Pred_Negative"]
)
safe_display(cm_logreg)

accuracy = accuracy_score(y_test, y_pred_test)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred_test, average="weighted", zero_division=0)
logreg_metrics = pd.DataFrame([{
    "method": "TF-IDF + Logistic Regression",
    "accuracy": round(accuracy, 4),
    "precision_weighted": round(precision, 4),
    "recall_weighted": round(recall, 4),
    "f1_weighted": round(f1, 4),
    "jumlah_test": len(y_test)
}])
safe_display(logreg_metrics)

MODEL_PATH = MODEL_DIR / "tfidf_logistic_regression_model.joblib"
joblib.dump(logreg_model, MODEL_PATH)
print("Model tersimpan:", MODEL_PATH)


Jumlah data berlabel: 1829
Distribusi label training/evaluasi:


,label_true,count
0,Positive,765
1,Neutral,657
2,Negative,407


Classification Report - Logistic Regression Test Set
              precision    recall  f1-score   support

    Positive       0.66      0.75      0.71       153
     Neutral       0.63      0.52      0.57       132
    Negative       0.74      0.77      0.75        81

    accuracy                           0.67       366
   macro avg       0.68      0.68      0.68       366
weighted avg       0.67      0.67      0.67       366



,Pred_Positive,Pred_Neutral,Pred_Negative
Actual_Positive,115,31,7
Actual_Neutral,48,69,15
Actual_Negative,10,9,62


,method,accuracy,precision_weighted,recall_weighted,f1_weighted,jumlah_test
0,TF-IDF + Logistic Regression,0.6721,0.6695,0.6721,0.6678,366


Model tersimpan: /content/outputs/models/tfidf_logistic_regression_model.joblib


## 4. Export hasil Logistic Regression

In [4]:

OUTPUT_CSV = SENTIMENT_DIR / "hasil_textblob_logreg_sentiment.csv"
OUTPUT_EXCEL = SENTIMENT_DIR / "hasil_textblob_logreg_sentiment.xlsx"

df.to_csv(OUTPUT_CSV, index=False)
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Prediction", index=False)
    logreg_metrics.to_excel(writer, sheet_name="Metrics", index=False)
    cm_logreg.to_excel(writer, sheet_name="Confusion_Matrix")

print("CSV tersimpan:", OUTPUT_CSV)
print("Excel tersimpan:", OUTPUT_EXCEL)

from google.colab import files
files.download(str(OUTPUT_CSV))


CSV tersimpan: /content/outputs/sentiment/hasil_textblob_logreg_sentiment.csv
Excel tersimpan: /content/outputs/sentiment/hasil_textblob_logreg_sentiment.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>